**Step 1: Object-Oriented Setup**
1. Import `@dataclass`.
2. Create a `@dataclass` named `SalesPipeline`.
3. Add two attributes: `pipeline_name` (str) and `raw_data` (list).
4. Define `__post_init__(self)` to automatically convert `pipeline_name` to uppercase.

**Step 2: Dynamic Functions (`*args`)**
1. Inside the class, define a method `ingest_batches(self, *args)`. 
2. The `*args` will be multiple lists of transaction dictionaries. Loop through `args` and extend `self.raw_data` with these new lists.

**Step 3: Comprehensions (List & Dict)**
1. Define a method `extract_successful_sales(self)`.
2. First, use a **list comprehension** to filter `self.raw_data`, keeping only records where the `"status"` is exactly `"success"`.
3. Then, use a **dictionary comprehension** on that filtered list to return a dictionary where the key is `"txn_id"` and the value is `"amount"`.

**Step 4: Error Handling (`try/except`)**
1. Define a method `calculate_average_sale(self)`.
2. Inside this method, call `self.extract_successful_sales()` to get your clean dictionary.
3. Use a `try/except` block to calculate the average sale amount (`sum` of values divided by the `len` of values).
4. Catch a `ZeroDivisionError` (if there are no successful sales) and print `"Warning: No successful transactions to average."`
5. Catch a generic `Exception` (in case a string gets passed as an amount instead of a number) and print the specific error message. Return the average (or 0 if it fails).

In [0]:
from dataclasses import dataclass

@dataclass
class SalesPipeline:
    pipeline_name: str
    raw_data: list
    
    def __post_init__(self):
        # Step 1: Format attribute on creation
        self.pipeline_name = self.pipeline_name.upper()
        
    def ingest_batches(self, *args):
        # Step 2: Handle dynamic positional arguments
        for batches in args:
            self.raw_data.extend(batches)
            print(f"ingested {len(batches)} batches " )    
    def extract_successful_sales(self):
        # Step 3a: List Comprehension with filtering
        # Note: using .get() prevents KeyError if "status" is missing from a corrupt record
        clean_data = [x for x in self.raw_data if x.get("status") == "success"]
        
        # Step 3b: Dict Comprehension for transformation
        better_data =  {y["txn_id"]: y["amount"] for y in clean_data}
        return better_data
        
    def calculate_average_sale(self):
        clean_sale = self.extract_successful_sales()
        sale_amount = list(clean_sale.values())
        # Step 4: Error Handling
        try:
            avg_sale = sum(sale_amount) / len(sale_amount)
            print(f"[{self.pipeline_name}]Average Sale: {avg_sale:.2f}")
            return avg_sale
        except ZeroDivisionError:
            print(f"Warning: No successful transactions to average.")
            return 0.0
        except Exception as e:
            print(f"Error: {e}")
            return 0.0

In [0]:
# Initialize empty pipeline
pipeline = SalesPipeline("shopee_daily_ingest", [])

# Mock Data Batches
batch_1 = [
    {"txn_id": "T001", "amount": 150.50, "status": "success"},
    {"txn_id": "T002", "amount": 20.00, "status": "failed"}
]
batch_2 = [
    {"txn_id": "T003", "amount": 300.00, "status": "success"},
    {"txn_id": "T004", "amount": "invalid_string", "status": "success"} # This will trigger the Exception!
]

# Test 1: Ingest Data using *args
pipeline.ingest_batches(batch_1, batch_2)

# Test 2: Calculate Average (Will fail due to "invalid_string" in amount)
pipeline.calculate_average_sale()

# Test 3: Test ZeroDivisionError handling by resetting data to only failed transactions
pipeline.raw_data = [{"txn_id": "T005", "amount": 100, "status": "failed"}]
pipeline.calculate_average_sale()

ingested 2 batches 
ingested 2 batches 
Error: unsupported operand type(s) for +: 'float' and 'str'


0.0